# AICG3D T4 傻瓜式部署：拉取自己的仓库 -> 下载模型 -> 启动 ComfyUI

**这份 Notebook 只做一件事：把 ComfyUI + 修复过的 AICG3D 稳稳地开起来。**
以后每次使用方法固定不变：

1. 在 Kaggle 上新建私人草稿，把这份 Notebook（仓库里的 `kaggle_t4.ipynb`）贴进来，选 GPU T4x2、Internet 开。
2. 按顺序从第 1 格跑到第 5 格，不要跳格。
3. 第 5 格会给出 Cloudflare 隧道 URL，打开它就是 ComfyUI，右键菜单里找 AICG3D 分类。
4. 在 ComfyUI 里导入 13 号无限分段工作流（仓库 `workflows/` 里），上传图片，核对时长、步数、分辨率、种子，点 Queue。
5. 生成期间每 20 分钟左右跑一次第 6 格核查，Kaggle 超过 40 分钟无改动会弹“你还在吗”并可能让会话离线。

**版本全部锁死，不拉最新、不 git pull：** ComfyUI commit ab08f7b2caa6d4007ecde3762116c6193613f43b、
torch 2.11.0+cu130、SageAttention2 2.1.2 sm75 轮子、comfy-kitchen 0.2.40、av 19.0.2。
Sage 只有验证通过才开，否则自动回落 SDPA，不会再刷 PassManager 错误。
隧道是 Cloudflare Quick Tunnel，无鉴权，URL 不要转发。不要在 Notebook 里写任何密码、API Key、QQ 凭证。


## 第 1 格说明：配置（只改这一格）

- `VRAM_MODE`：默认 `normalvram`，不要默认 lowvram；OOM 再降档并记录。
- `MODEL_SET`：默认 `full`，FL2VA 出现两次是故意的绕 bug 做法，不要去重。
- `WORKDIR`：Kaggle 默认 `/kaggle/temp/h3_project`，模型和 ComfyUI 都在这里，会话离线会消失，生成完务必下载到本地。


In [ ]:
# 第 1 格：配置 + torch 2.11.0+cu130（Sage 轮子只对应这个版本）
import os, sys, subprocess
from pathlib import Path

VRAM_MODE = "normalvram"  # "normalvram" / "highvram" / "lowvram"
MODEL_SET = "full"        # 本 Notebook 固定 full，不提供 ref2va_only 简化档
WORKDIR = Path("/kaggle/temp/h3_project")
COMFY = WORKDIR / "ComfyUI"
LOGDIR = WORKDIR / "logs"
for p in [WORKDIR, LOGDIR]:
    p.mkdir(parents=True, exist_ok=True)

def run(cmd, check=True):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=check)

assert VRAM_MODE in ("normalvram", "highvram", "lowvram")
print("配置：", {"VRAM_MODE": VRAM_MODE, "MODEL_SET": MODEL_SET, "WORKDIR": str(WORKDIR)})
run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cu130", "torchvision==0.26.0+cu130", "torchaudio==2.11.0+cu130", "--index-url", "https://download.pytorch.org/whl/cu130"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "huggingface-hub>=1.5.0,<2.0", "hf_transfer", "ipywidgets", "requests", "packaging"])
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import torch
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda, "可用:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cuda.sdp_kernel(enable_flash=False, enable_mem_efficient=True, enable_math=False)
print("第 1 格完成。")


## 第 2 格说明：固定版 ComfyUI + 本仓库 AICG3D + fp16_fix

- ComfyUI 不拉 master，只拉 commit ab08f7b2caa6d4007ecde3762116c6193613f43b 的 tarball，
  已经是这个版本就跳过，不重复下载。
- AICG3D 不是从上游 JGRFW 拉，而是从本仓库（你自己的仓库）拷到
  `ComfyUI/custom_nodes/comfyui-AICG3D`。**本格会自动 clone 本仓库，不用事先手动 clone：**
  如果 `/kaggle/working/aicg3d-t4-repo` 里没有 `aicg3d_h3easy/accel.py`，
  本格会先 `git clone https://github.com/phao00918/comfyui-AICG3D-T4.git` 到那个目录，
  再继续；只有 clone 本身失败（比如没开 Internet、网络不通）才会报错停住。
- fp16_fix 校验：>1000 字节且含 MiniMaxH3Model，否则不启动。


In [ ]:
# 第 2 格：固定版 ComfyUI + 本仓库 AICG3D + fp16_fix
import os, sys, subprocess, urllib.request, shutil
from pathlib import Path

WORKDIR = Path("/kaggle/temp/h3_project")
COMFY = WORKDIR / "ComfyUI"
LOGDIR = WORKDIR / "logs"
COMFYUI_SHA = "ab08f7b2caa6d4007ecde3762116c6193613f43b"
# 本仓库在 Kaggle 上的位置：优先 /kaggle/working/aicg3d-t4-repo，其次默认 clone 目录名
REPO_CANDIDATES = [Path("/kaggle/working/aicg3d-t4-repo"), Path("/kaggle/working/comfyui-AICG3D-T4")]
REPO_DIR = next((p for p in REPO_CANDIDATES if (p / "aicg3d_h3easy" / "accel.py").exists()), None)
if REPO_DIR is None:
    # 之前版本这里直接报错，是本次报错的根因：改为本格自动 clone，不用事先手动 clone。
    # 仓库是公开的，Kaggle 上 Internet 开着就不用 token。
    target = Path("/kaggle/working/aicg3d-t4-repo")
    if target.exists() and not (target / "aicg3d_h3easy" / "accel.py").exists():
        print("目标目录已存在但不是完整仓库，先清空再重新 clone：", target)
        shutil.rmtree(target)
    print("本仓库还没拉取，正在自动 clone：", target)
    r = subprocess.run(["git", "clone", "https://github.com/phao00918/comfyui-AICG3D-T4.git", str(target)],
                       capture_output=True, text=True)
    print(r.stdout)
    print(r.stderr)
    if r.returncode != 0 or not (target / "aicg3d_h3easy" / "accel.py").exists():
        raise RuntimeError("自动 clone 失败。请先确认 Kaggle 已开 Internet，再重跑本格；"
                           "如果反复失败，再手动跑一格：!git clone https://github.com/phao00918/comfyui-AICG3D-T4.git /kaggle/working/aicg3d-t4-repo，"
                           "并把 clone 的完整输出发回来看。")
    REPO_DIR = target
print("本仓库：", REPO_DIR)

def sh(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

marker = WORKDIR / ".comfyui_pinned_sha"
if marker.exists() and marker.read_text().strip() == COMFYUI_SHA and (COMFY / "main.py").exists():
    print("ComfyUI 已是锁死版本，跳过下载。")
else:
    if COMFY.exists():
        shutil.rmtree(COMFY)
    sh(f"curl -L -o /tmp/comfyui.tar.gz https://codeload.github.com/comfyanonymous/ComfyUI/tar.gz/{COMFYUI_SHA}")
    sh(f"mkdir -p {WORKDIR}/comfyui_tmp && tar -xzf /tmp/comfyui.tar.gz -C {WORKDIR}/comfyui_tmp && mv {WORKDIR}/comfyui_tmp/ComfyUI-{COMFYUI_SHA} {COMFY} && rmdir {WORKDIR}/comfyui_tmp")
    marker.write_text(COMFYUI_SHA)
    print("ComfyUI 已拉到锁死版本：", COMFYUI_SHA)

sh(f"{sys.executable} -m pip install -r {COMFY}/requirements.txt")
sh(f"{sys.executable} -m pip install comfy-aimdo==0.6.0 torchsde comfy-kitchen==0.2.40 av==19.0.2")

dest = COMFY / "custom_nodes" / "comfyui-AICG3D"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(REPO_DIR, dest, ignore=shutil.ignore_patterns(".git", "__pycache__"))
print("AICG3D 已从本仓库拷到：", dest)

PATCH_URL = "https://raw.githubusercontent.com/Amduraznak/minimax-h3-fp16-fix/main/minimax_h3_fp16_fix.py"
PATCH_FILE = COMFY / "custom_nodes" / "minimax_h3_fp16_fix.py"
urllib.request.urlretrieve(PATCH_URL, "/tmp/minimax_h3_fp16_fix.py")
data = Path("/tmp/minimax_h3_fp16_fix.py").read_bytes()
assert len(data) > 1000 and b"MiniMaxH3Model" in data, "fp16_fix 校验失败"
PATCH_FILE.write_bytes(data)
print("fp16_fix OK:", PATCH_FILE, len(data), "bytes")

sh(f"{sys.executable} -m pip install -r {dest}/requirements.txt", check=False)
sh(f"{sys.executable} -m pip install imageio-ffmpeg psutil requests aiohttp", check=False)
print("第 2 格完成。")


## 第 3 格说明：Sage 轮子 + 验证（验证不过就回落 SDPA，不中断）

- 轮子按 Python 版本自动选 cp312/cp313，torch 2.11.0+cu130，`--no-deps` 安装。
- 验证脚本在 H3 形状下跑 SDPA 对比，cos >= 0.999 且 seq=4096 提速 >= 1.2x 才写标记。
- 标记写在 `ComfyUI/custom_nodes/comfyui-AICG3D/t4_sage_sm75_verified.json`，
  accel.py 只认这个标记，验证不过时 AICG3D 会自动走 SDPA。


In [ ]:
# 第 3 格：装 sm75 Sage 轮子并验证
import sys, subprocess, urllib.request
from pathlib import Path

COMFY = Path("/kaggle/temp/h3_project") / "ComfyUI"
AICG = COMFY / "custom_nodes" / "comfyui-AICG3D"
MARKER = AICG / "t4_sage_sm75_verified.json"
if MARKER.exists():
    MARKER.unlink()
CP_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
print("Python tag:", CP_TAG)
url = f"https://github.com/netrunner-exe/SageAttention2-Colab-Wheels/raw/main/optimized_wheels/colab-torch-{CP_TAG}-2.11.0%2Bcu130/sageattention-2.1.2-{CP_TAG}-{CP_TAG}-linux_x86_64.whl"
print("轮子地址：", url)
try:
    urllib.request.urlretrieve(url, "/tmp/sage_wheel.whl")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "/tmp/sage_wheel.whl"], check=True)
    r = subprocess.run([sys.executable, str(AICG / "scripts" / "verify_sage_sm75.py"), "--marker", str(MARKER)])
    if r.returncode == 0 and MARKER.exists():
        print("Sage 验证通过，启动时会带 --use-sage-attention。")
    else:
        print("警告：Sage 验证没通过，启动时不带 --use-sage-attention，走 SDPA。")
except Exception as e:
    print("警告：Sage 轮子下载/安装失败，走 SDPA：", repr(e))
print("第 3 格完成。")


In [ ]:
# 第 4 格：下载模型（full 清单，FL2VA 两次是故意的，不要去重）
import os, sys, subprocess, shutil
from pathlib import Path

WORKDIR = Path("/kaggle/temp/h3_project")
COMFY = WORKDIR / "ComfyUI"
MODEL_ROOT = COMFY / "models"
for folder in ["diffusion_models", "text_encoders", "vae", "loras", "embeddings"]:
    (MODEL_ROOT / folder).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORKDIR / "hf_home")
os.environ["HF_HUB_CACHE"] = str(WORKDIR / "hf_home" / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
FILES = [
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "vae/minimax_h3_video_vae_fp16.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
    "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
    "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
]
print("文件数（含 FL2VA 重复）：", len(FILES))
if shutil.which("hf"):
    cmd = ["hf", "download", "Comfy-Org/MiniMax-H3", "--local-dir", str(MODEL_ROOT), "--include", *FILES]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    raise SystemExit("hf CLI missing")
ok = True
seen = set()
for f in FILES:
    if f in seen:
        continue
    seen.add(f)
    p = MODEL_ROOT / f
    size = p.stat().st_size if p.exists() else 0
    print(f"  {f}: {size} bytes {'OK' if size > 100_000_000 else 'FAIL'}")
    ok = ok and size > 100_000_000
if not ok:
    raise RuntimeError("模型校验失败：有文件缺失或过小，不要启动 ComfyUI。")
print("模型校验通过。第 4 格完成。")


In [ ]:
# 第 5 格：启动 ComfyUI + 隧道（Sage 标记通过才带 --use-sage-attention）
import subprocess, time, socket, shutil
from pathlib import Path

WORKDIR = Path("/kaggle/temp/h3_project")
COMFY = WORKDIR / "ComfyUI"
LOGDIR = WORKDIR / "logs"
MARKER = COMFY / "custom_nodes" / "comfyui-AICG3D" / "t4_sage_sm75_verified.json"
VRAM_MODE = "normalvram"
args = [__import__("sys").executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"]
if VRAM_MODE == "highvram":
    args.append("--highvram")
elif VRAM_MODE == "lowvram":
    args.append("--lowvram")
args += ["--disable-pinned-memory", "--preview-method", "none", "--fp16-unet"]
if MARKER.exists():
    args.append("--use-sage-attention")
    print("Sage 标记存在，启动带 --use-sage-attention。")
else:
    print("Sage 标记不存在，启动不带 --use-sage-attention，走 SDPA。")
print("启动命令：", " ".join(args))
logfile = open(LOGDIR / "comfyui.log", "a", buffering=1)
proc = subprocess.Popen(args, cwd=str(COMFY), stdout=logfile, stderr=subprocess.STDOUT, start_new_session=True)
print("ComfyUI PID:", proc.pid)
for i in range(36):
    time.sleep(5)
    try:
        s = socket.create_connection(("127.0.0.1", 8188), timeout=2)
        s.close()
        print(f"ComfyUI 已就绪（约 {(i+1)*5} 秒）。")
        break
    except Exception:
        if i % 6 == 0:
            print(f"等待中... {(i+1)*5}s")
else:
    print("警告：180 秒内端口 8188 未就绪，请看日志：", LOGDIR / "comfyui.log")
print("\n启动 Cloudflare 隧道（Quick Tunnel，无鉴权，URL 不要转发）：")
if not shutil.which("cloudflared"):
    subprocess.run(["bash", "-lc", "curl -L -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/download/2024.10.0/cloudflared-linux-amd64.deb && dpkg -i /tmp/cloudflared.deb || true"], check=False)
if shutil.which("cloudflared"):
    tunnel_log = open(LOGDIR / "cloudflared.log", "a", buffering=1)
    tp = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    print("cloudflared PID:", tp.pid)
    time.sleep(20)
    try:
        print((LOGDIR / "cloudflared.log").read_text()[-2000:])
    except Exception as e:
        print("读隧道日志失败：", repr(e))
else:
    print("cloudflared 仍不可用，请手动安装固定版本 2024.10.0。")
print("\n第 5 格完成。打开隧道 URL 进入 ComfyUI，右键找 AICG3D，导入 13 号工作流。")


In [ ]:
# 第 6 格：核查队列 + 保活（生成期间每 20 分钟跑一次，不提交生成）
import urllib.request, json, time
from pathlib import Path
COMFY = Path("/kaggle/temp/h3_project") / "ComfyUI"
OUT_DIR = COMFY / "output" / "video"
print("核查时间：", time.strftime("%Y-%m-%d %H:%M:%S"))
try:
    with urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5) as r:
        q = json.loads(r.read().decode())
    print("队列 running:", len(q.get("queue_running", [])), "pending:", len(q.get("queue_pending", [])))
except Exception as e:
    print("队列接口不可用：", repr(e))
if OUT_DIR.exists():
    videos = sorted(OUT_DIR.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    print("输出视频数：", len(videos))
    for v in videos[-5:]:
        print(" -", v.name, v.stat().st_size, "bytes")
else:
    print("输出目录尚不存在。")
print("本格只核查和保活，不提交生成。生成完请把视频下载到本地，不要只留在 Kaggle 上。")
